[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [15]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [16]:
import torch

In [42]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    logits = logits / temperature
    if top_k > 0:
      k = min(top_k, logits.shape[-1])
      threshold = torch.topk(logits, k, dim=-1).values[..., -1:]
      logits = logits.masked_fill(logits < threshold, float('-inf'))
    if top_p < 1.0:
      prob = torch.softmax(logits, dim=-1)
      sorted_prob, sorted_idx = torch.sort(prob, dim=-1, descending=True)
      cumsum = torch.cumsum(sorted_prob, dim=-1)
      sorted_mask = (cumsum - sorted_prob) >= top_p
      mask = torch.zeros_like(sorted_mask).scatter(-1, sorted_idx, sorted_mask)
      logits = logits.masked_fill(mask, float('-inf'))
    probs = torch.softmax(logits, dim=-1)
    samples = torch.multinomial(probs, num_samples=1)
    return samples.squeeze(-1).item()
    # values, indices = torch.topk(logits, top_k, dim=-1)
    # logits.masked_fill(logits < values[-1], float('-inf'))
    # # print(logits)
    # prob = torch.softmax(logits, dim=-1)
    # sorted, indices = torch.sort(prob, dim=-1, descending=True, )
    # cumsum = torch.cumsum(sorted, dim=-1)
    # selected = torch.masked_select(indices, cumsum < top_p) #indices, cumsum<top_p
    pass  # temperature, top-k filter, top-p filter, sample

In [43]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [44]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (6.0ms)
  ✅ [2/4] Low temperature concentrates (6.9ms)
  ✅ [3/4] All tokens reachable (no filtering) (174.8ms)
  ✅ [4/4] Returns valid index (5.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (192.7ms total)
  Progress saved. Run status() to see your dashboard.

